In [18]:
import pandas as pd
import json
import time
import requests
import os

OPENROUTER_API_KEY = os.getenv("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")  

url = "https://openrouter.ai/api/v1/chat/completions"

Models evaluating:
- DeepSeek — DeepSeek V3
- Anthropic — Claude 3.5
- Google — Gemini 2.5 Flash
- Meta — Llama 3.1 70B Instruct
- Mistral - Mistral Large

In [ ]:
eval_df = pd.read_csv("yoruba_proverbs_eval_final.csv", encoding="utf-8-sig")
print(eval_df.shape)

MODEL_KEYS = ["openai", "claude", "gemini", "mistral", "llama"]

for m in MODEL_KEYS:
    eval_df[f"{m}_translation"] = ""
    eval_df[f"{m}_interpretation"] = ""
    eval_df[f"{m}_raw"] = ""     # store raw model output (debug)
    eval_df[f"{m}_error"] = ""   # store errors (rate limit, json issues, etc.)

eval_df.head(2)
eval_df.columns.tolist()

(2272, 10)


['entry_id',
 'part_id',
 'part_code',
 'part_theme',
 'letter',
 'yoruba',
 'english_literal',
 'interpretation_gold',
 'quality_flag',
 'has_yoruba_diacritics',
 'openai_translation',
 'openai_interpretation',
 'openai_raw',
 'openai_error',
 'claude_translation',
 'claude_interpretation',
 'claude_raw',
 'claude_error',
 'gemini_translation',
 'gemini_interpretation',
 'gemini_raw',
 'gemini_error',
 'mistral_translation',
 'mistral_interpretation',
 'mistral_raw',
 'mistral_error',
 'llama_translation',
 'llama_interpretation',
 'llama_raw',
 'llama_error']

In [ ]:
PROMPT = """You are a Yoruba language and culture expert.

Given this Yoruba proverb, return VALID JSON ONLY with exactly these keys:
{
  "translation": string,        // literal English translation (surface meaning)
  "interpretation": string      // meaning/lesson in 2–4 sentences
}

Rules:
- Output JSON only. No markdown. No extra keys.
- Do not invent details not supported by the proverb.
- Keep interpretation concise and culturally grounded.
- Translation must be literal (surface meaning).
- Interpretation must explain the meaning in 2–4 sentences.

Proverb: "{proverb}"
"""

In [14]:
import json

def parse_json_strict(text: str) -> dict:
    """
    Safely extracts and validates JSON output from an LLM response.
    Ensures it contains exactly:
    - translation
    - interpretation
    """

    # 1️⃣ Clean whitespace
    t = (text or "").strip()

    # 2️⃣ Extract JSON block if model added extra text
    if not t.startswith("{"):
        start = t.find("{")
        end = t.rfind("}")
        if start != -1 and end != -1 and end > start:
            t = t[start:end+1]

    # 3️⃣ Convert string → Python dict
    obj = json.loads(t)

    # 4️⃣ Validate structure
    if not isinstance(obj, dict):
        raise ValueError("Output JSON is not an object.")

    if "translation" not in obj or "interpretation" not in obj:
        raise ValueError(
            f"Missing required keys. Found keys: {list(obj.keys())}"
        )

    return obj

In [15]:
# Perfect JSON
print(parse_json_strict('{"translation":"Hello","interpretation":"Meaning"}'))

# JSON wrapped in extra text
print(parse_json_strict('Sure! {"translation":"Hi","interpretation":"Lesson"} Thanks!'))

{'translation': 'Hello', 'interpretation': 'Meaning'}
{'translation': 'Hi', 'interpretation': 'Lesson'}


In [17]:
sample = eval_df.head(10).copy()
sample["yoruba"].tolist()[:3]

['Àgbéré àwòdì ní ńní òun ó jẹ ìgbín.',
 'A kì í kórira ọ̀fọ́n-ọ̀n ká finá bọ ahéré.',
 'A kì í bọ òrìṣà lójú ọ̀fọ́n-ọ̀n; bó bá dalẹ́ a máa tú pẹpẹ.']

In [21]:
def call_model(model_key, proverb_text):
    """
    Return dict: {"translation": "...", "interpretation": "..."}
    """
    raise NotImplementedError

CHECKPOINT_EVERY = 25

for i, row in enumerate(eval_df.itertuples()):
    proverb = getattr(row, "yoruba")
    idx = row.Index

    for m in MODEL_KEYS:
        if eval_df.at[idx, f"{m}_translation"] != "":
            continue

        try:
            out = call_model(m, proverb)
            eval_df.at[idx, f"{m}_translation"] = out["translation"]
            eval_df.at[idx, f"{m}_interpretation"] = out["interpretation"]
        except Exception as e:
            eval_df.at[idx, f"{m}_error"] = str(e)

    if (i+1) % CHECKPOINT_EVERY == 0:
        eval_df.to_csv("eval_running_checkpoint.csv", index=False, encoding="utf-8-sig")
        print("Checkpoint saved:", i+1)
        time.sleep(0.5)

Checkpoint saved: 25
Checkpoint saved: 50
Checkpoint saved: 75
Checkpoint saved: 100
Checkpoint saved: 125
Checkpoint saved: 150
Checkpoint saved: 175
Checkpoint saved: 200
Checkpoint saved: 225
Checkpoint saved: 250
Checkpoint saved: 275
Checkpoint saved: 300
Checkpoint saved: 325
Checkpoint saved: 350
Checkpoint saved: 375
Checkpoint saved: 400
Checkpoint saved: 425
Checkpoint saved: 450
Checkpoint saved: 475
Checkpoint saved: 500
Checkpoint saved: 525
Checkpoint saved: 550
Checkpoint saved: 575
Checkpoint saved: 600
Checkpoint saved: 625
Checkpoint saved: 650
Checkpoint saved: 675
Checkpoint saved: 700
Checkpoint saved: 725
Checkpoint saved: 750
Checkpoint saved: 775
Checkpoint saved: 800
Checkpoint saved: 825
Checkpoint saved: 850
Checkpoint saved: 875
Checkpoint saved: 900
Checkpoint saved: 925
Checkpoint saved: 950
Checkpoint saved: 975
Checkpoint saved: 1000
Checkpoint saved: 1025
Checkpoint saved: 1050
Checkpoint saved: 1075
Checkpoint saved: 1100
Checkpoint saved: 1125
Checkpo

In [20]:
#test
import requests
import os

OPENROUTER_API_KEY = os.getenv("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")  

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "http://localhost",          # can be anything
    "X-OpenRouter-Title": "Yoruba Proverb Eval",  # can be anything
}


model = "google/gemini-2.5-flash"  # <- one of your chosen models

proverb = "Bí a bá rí ẹni tí ó ń wẹ̀ ní ojú-ọ̀nà, a kì í béèrè bó ṣe rí omi."

prompt = f"""
Return JSON ONLY with keys:
- literal_translation: string (1 sentence)
- interpretation: string (2-3 sentences max)
- confidence: number (0.0 to 1.0)

Yoruba proverb: {proverb}
""".strip()

payload = {
    "model": model,
    "messages": [{"role": "user", 
                  "content": prompt}],
    "temperature": 0.2,
    "top_p": 0.9
}

r = requests.post(url, headers=headers, json=payload, timeout=60)
print("Status:", r.status_code)
print(r.text)  # raw output so you can debug

# If status 200, also print just the model reply
if r.status_code == 200:
    j = r.json()
    print("\nMODEL OUTPUT:\n", j["choices"][0]["message"]["content"])
    print("\nUSAGE:\n", j.get("usage", {}))

Status: 401
{"error":{"message":"Missing Authentication header","code":401}}
